# 01 — BERT Tokenization Basics for CTI Text

**Goal:** Understand how BERT breaks cyber threat intelligence (CTI) text into tokens before we do anything with NER or classification.

## Why tokenization matters first

BERT doesn't read words — it reads **tokens**. A single word like `X-Agent` or `CVE-2017-0199` may be split into several pieces (called *subwords* or *WordPieces*). This has big consequences:

- **For NER:** if `X-Agent` becomes 3 tokens, you need to decide how to align 1 label to 3 tokens.
- **For classification:** long threat reports get truncated at 512 tokens — you need to know how much text that actually is.
- **For CTI specifically:** threat actor names (APT28), malware (X-Agent), CVEs, hashes, and domains often tokenize in surprising ways because they weren't common in BERT's training data.

## What you'll learn in this notebook

1. Load a BERT tokenizer
2. Tokenize a real CTI sentence and inspect the output
3. Understand special tokens: `[CLS]`, `[SEP]`, `[PAD]`
4. Understand `input_ids`, `attention_mask`, and `token_type_ids`
5. See how CTI-specific terms (APT28, CVE-IDs, malware names) get tokenized
6. Decode tokens back to text

## Step 0 — Install dependencies

We need `transformers` (Hugging Face library that gives us BERT) and `torch` (the deep learning backend BERT runs on).

Run the cell below **once**. If you already have them installed, it will just confirm.

In [ ]:
# Uncomment the next line if running for the first time
# !pip install transformers torch

## Step 1 — Load the tokenizer

We'll start with `bert-base-uncased`. It's the standard BERT model trained on general English text.

Later we'll compare this with CTI-domain models like `SecureBERT` to see the difference.

> **Note:** `uncased` means the tokenizer lowercases everything. `APT28` becomes `apt28`. This is often fine for classification but can hurt NER (threat actor names are proper nouns). We'll revisit this.

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(f"Tokenizer type: {type(tokenizer).__name__}")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Max input length: {tokenizer.model_max_length}")
print(f"Special tokens: {tokenizer.all_special_tokens}")

Tokenizer type: BertTokenizerFast
Vocabulary size: 30522
Max input length: 512
Special tokens: ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']


### What just happened?

- **Vocabulary size (~30,522):** BERT knows this many unique tokens. Anything not in the vocab gets broken into subwords.
- **Max input length (512):** BERT cannot process more than 512 tokens at once. Long reports must be truncated or chunked.
- **Special tokens:** These are markers BERT relies on — you'll see them in the next step.

## Step 2 — Tokenize a CTI sentence

Let's use a realistic threat intelligence sentence:

In [2]:
cti_sentence = "APT28 deployed X-Agent malware exploiting CVE-2017-0199."

tokens = tokenizer.tokenize(cti_sentence)
print("Tokens:")
print(tokens)

Tokens:
['apt', '##28', 'deployed', 'x', '-', 'agent', 'mal', '##ware', 'exploit', '##ing', 'cv', '##e', '-', '2017', '-', '01', '##9', '##9', '.']


### Read the output carefully

You'll see something like:
```
['apt', '##28', 'deployed', 'x', '-', 'agent', 'malware', 'exploiting', 'cve', '-', '2017', '-', '01', '##99', '.']
```

Key things to notice:

| Observation | Why it matters |
|---|---|
| `APT28` → `apt` + `##28` | `##` means "this is a continuation of the previous token". One word, two tokens. |
| `X-Agent` → `x`, `-`, `agent` | The hyphen is its own token. One malware name = 3 tokens. |
| `CVE-2017-0199` → 6 tokens | CVE IDs fragment heavily because they're not in general English vocab. |
| Everything is lowercase | Because we used the `uncased` model. |

**This is the subword-alignment problem.** If a human labels `APT28` as a `THREAT_ACTOR`, but BERT sees `apt` and `##28` as separate tokens, how do we assign labels? We'll solve this in a later notebook.

## Step 3 — Full encoding with special tokens

`tokenizer.tokenize()` only gives us the raw tokens. To actually feed text into BERT, we use `tokenizer()` which returns everything the model needs.

In [3]:
encoded = tokenizer(cti_sentence, return_tensors="pt")

print("Keys returned:", list(encoded.keys()))
print()
for key, value in encoded.items():
    print(f"{key}:")
    print(value)
    print()

Keys returned: ['input_ids', 'token_type_ids', 'attention_mask']

input_ids:
tensor([[  101, 26794, 22407,  7333,  1060,  1011,  4005, 15451,  8059, 18077,
          2075, 26226,  2063,  1011,  2418,  1011,  5890,  2683,  2683,  1012,
           102]])

token_type_ids:
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

attention_mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])



### The three outputs explained

**1. `input_ids`** — the tokens converted to integer IDs from BERT's vocabulary. This is what the model actually receives.

**2. `attention_mask`** — tells BERT which tokens are real (1) vs. padding (0). With a single sentence there's no padding yet, so it's all 1s.

**3. `token_type_ids`** — used when feeding BERT *two* sentences (e.g., question + answer). For a single sentence it's all 0s. Ignore for now.

## Step 4 — See the special tokens [CLS] and [SEP]

When you use `tokenizer()` (full encoding), BERT automatically adds two special tokens:

- `[CLS]` at the start — a "classification" token. Its final hidden state is used for **sequence classification**.
- `[SEP]` at the end — a "separator" token.

Let's decode `input_ids` back to tokens to see them:

In [4]:
input_ids = encoded["input_ids"][0]
tokens_with_special = tokenizer.convert_ids_to_tokens(input_ids)

print("Tokens with special tokens:")
for i, (tok_id, tok) in enumerate(zip(input_ids.tolist(), tokens_with_special)):
    print(f"  Position {i:2d} | ID {tok_id:5d} | Token: {tok}")

Tokens with special tokens:
  Position  0 | ID   101 | Token: [CLS]
  Position  1 | ID 26794 | Token: apt
  Position  2 | ID 22407 | Token: ##28
  Position  3 | ID  7333 | Token: deployed
  Position  4 | ID  1060 | Token: x
  Position  5 | ID  1011 | Token: -
  Position  6 | ID  4005 | Token: agent
  Position  7 | ID 15451 | Token: mal
  Position  8 | ID  8059 | Token: ##ware
  Position  9 | ID 18077 | Token: exploit
  Position 10 | ID  2075 | Token: ##ing
  Position 11 | ID 26226 | Token: cv
  Position 12 | ID  2063 | Token: ##e
  Position 13 | ID  1011 | Token: -
  Position 14 | ID  2418 | Token: 2017
  Position 15 | ID  1011 | Token: -
  Position 16 | ID  5890 | Token: 01
  Position 17 | ID  2683 | Token: ##9
  Position 18 | ID  2683 | Token: ##9
  Position 19 | ID  1012 | Token: .
  Position 20 | ID   102 | Token: [SEP]


Notice:
- Position 0 is `[CLS]`
- Last position is `[SEP]`
- Everything in between is our sentence

**Why this matters later:**
- For **classification**, we'll grab the output vector at position 0 (`[CLS]`) — it represents the whole sentence.
- For **NER**, we'll use the output vectors at each middle position — one label per token.

## Step 5 — Tokenize several CTI examples and compare

Let's see how different CTI patterns tokenize. Pay attention to which entities fragment the most.

In [5]:
cti_examples = [
    "APT28 is a Russian state-sponsored threat actor.",
    "The malware communicates with C2 server at 185.92.222.46.",
    "Emotet uses macro-enabled documents for initial access.",
    "Hash: d41d8cd98f00b204e9800998ecf8427e was observed in the attack.",
    "The threat actor exploited CVE-2021-44228 (Log4Shell).",
]

for sentence in cti_examples:
    tokens = tokenizer.tokenize(sentence)
    print(f"Sentence: {sentence}")
    print(f"  Tokens ({len(tokens)}): {tokens}")
    print()

Sentence: APT28 is a Russian state-sponsored threat actor.
  Tokens (11): ['apt', '##28', 'is', 'a', 'russian', 'state', '-', 'sponsored', 'threat', 'actor', '.']

Sentence: The malware communicates with C2 server at 185.92.222.46.
  Tokens (17): ['the', 'mal', '##ware', 'communicate', '##s', 'with', 'c2', 'server', 'at', '185', '.', '92', '.', '222', '.', '46', '.']

Sentence: Emotet uses macro-enabled documents for initial access.
  Tokens (12): ['em', '##ote', '##t', 'uses', 'macro', '-', 'enabled', 'documents', 'for', 'initial', 'access', '.']

Sentence: Hash: d41d8cd98f00b204e9800998ecf8427e was observed in the attack.
  Tokens (33): ['hash', ':', 'd', '##41', '##d', '##8', '##cd', '##9', '##8', '##f', '##00', '##b', '##20', '##4', '##e', '##9', '##80', '##0', '##9', '##9', '##8', '##ec', '##f', '##8', '##42', '##7', '##e', 'was', 'observed', 'in', 'the', 'attack', '.']

Sentence: The threat actor exploited CVE-2021-44228 (Log4Shell).
  Tokens (19): ['the', 'threat', 'actor', 'exp

### Things to observe

- **IP addresses** (`185.92.222.46`) break into many pieces. BERT has no concept of an IP.
- **Hashes** (MD5/SHA) become long sequences of subwords. They eat a lot of your 512-token budget.
- **Malware names** (`Emotet`, `Log4Shell`) often fragment.
- **Common words** (`malware`, `threat`, `actor`) are usually single tokens — they appeared enough in BERT's training data.

> **Takeaway:** General-purpose BERT is *not ideal* for CTI. This motivates domain-specific models like **SecureBERT** and **CyBERT**, which were pre-trained on cybersecurity text and handle these terms better.

## Step 6 — Decoding back to text

You can always go from IDs → tokens → text. This is useful for debugging and for displaying model outputs.

In [6]:
original = "APT28 deployed X-Agent malware exploiting CVE-2017-0199."
encoded = tokenizer(original)

# IDs -> string
decoded = tokenizer.decode(encoded["input_ids"])
print("Decoded (with special tokens):")
print(decoded)
print()

# Skip [CLS] and [SEP]
decoded_clean = tokenizer.decode(encoded["input_ids"], skip_special_tokens=True)
print("Decoded (clean):")
print(decoded_clean)

Decoded (with special tokens):
[CLS] apt28 deployed x - agent malware exploiting cve - 2017 - 0199. [SEP]

Decoded (clean):
apt28 deployed x - agent malware exploiting cve - 2017 - 0199.


Notice that `apt28` comes back as `apt28` (joined) but the casing is lost (it's lowercase). Tokenization is **not perfectly reversible** — the `uncased` model throws away case information forever.

## Your turn — exercises

Try these in a new cell below to cement what you learned:

1. Tokenize a sentence with a SHA-256 hash. Count the tokens. How much of BERT's 512-token budget would 10 hashes consume?
2. Try loading `bert-base-cased` instead of `bert-base-uncased`. Does `APT28` still lowercase? Why might cased matter for NER?
3. Pick a paragraph from a real threat report (e.g., a Mandiant or CrowdStrike blog). Tokenize it. How many tokens is it? Does it fit in 512?
4. Find a word that produces the most subword fragments in a CTI context. (Try unusual malware or campaign names.)

In [42]:
# Your experiments here

from transformers import AutoTokenizer

tokenizer_uncased = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer_cased = AutoTokenizer.from_pretrained("bert-base-cased")

In [44]:
cti_examples = [
    "A highly targeted spear-phishing campaign was observed delivering a malicious attachment named Q4_Financial_Statement_Review_Confidential.xlsx.exe, which was carefully crafted to impersonate a legitimate corporate financial report and was distributed through compromised email accounts belonging to a regional logistics organization in order to increase credibility and bypass user suspicion.",
    
    "The malware demonstrated advanced persistence techniques by injecting its payload into legitimate Windows processes such as explorer.exe and svchost.exe, allowing it to blend seamlessly with normal system activity while evading detection mechanisms deployed by endpoint security solutions across the infected enterprise environment.",
    
    "The primary malicious binary was identified with the SHA-256 hash 9c7d2f4a6b8e3c1d5f0a9e2d4c6b7a1f8e3d9c0b5a7f6e2d1c3b4a5f67890abc, which was later confirmed to be part of a multi-stage infection chain involving in-memory execution, dynamic payload unpacking, and obfuscated code designed to hinder static and dynamic analysis efforts by security researchers.",
    
    "During analysis, the malware was found to implement API hashing techniques and custom resolution mechanisms for Windows API functions, thereby avoiding traditional import table-based detection and significantly increasing the difficulty of reverse engineering efforts by automated sandboxing and antivirus systems.",
    
    "The overall infection chain followed a sophisticated multi-stage execution model in which an initial loader was responsible for decrypting and unpacking a second-stage payload directly into memory, thereby avoiding disk writes, reducing forensic artifacts, and ensuring that malicious components remained active only in volatile system memory for stealthier operation."
]

In [45]:
# Uncased

for sentence in cti_examples:
    tokens = tokenizer_uncased.tokenize(sentence)
    print(f"Sentence: {sentence}")
    print(f"  Tokens ({len(tokens)}): {tokens}")
    print()

Sentence: A highly targeted spear-phishing campaign was observed delivering a malicious attachment named Q4_Financial_Statement_Review_Confidential.xlsx.exe, which was carefully crafted to impersonate a legitimate corporate financial report and was distributed through compromised email accounts belonging to a regional logistics organization in order to increase credibility and bypass user suspicion.
  Tokens (69): ['a', 'highly', 'targeted', 'spear', '-', 'phi', '##shing', 'campaign', 'was', 'observed', 'delivering', 'a', 'malicious', 'attachment', 'named', 'q', '##4', '_', 'financial', '_', 'statement', '_', 'review', '_', 'confidential', '.', 'xl', '##s', '##x', '.', 'ex', '##e', ',', 'which', 'was', 'carefully', 'crafted', 'to', 'imp', '##erson', '##ate', 'a', 'legitimate', 'corporate', 'financial', 'report', 'and', 'was', 'distributed', 'through', 'compromised', 'email', 'accounts', 'belonging', 'to', 'a', 'regional', 'logistics', 'organization', 'in', 'order', 'to', 'increase', 'c

In [27]:
# # Cased
# tokens = tokenizer_cased.tokenize(cti_sentence)
# for sentence in cti_examples:
#     tokens = tokenizer_cased.tokenize(sentence)
#     print(f"Sentence: {sentence}")
#     print(f"  Tokens ({len(tokens)}): {tokens}")
#     print()

Sentence: APT28 is a Russian state-sponsored threat actor.
  Tokens (11): ['apt', '##28', 'is', 'a', 'russian', 'state', '-', 'sponsored', 'threat', 'actor', '.']

Sentence: The malware communicates with C2 server at 185.92.222.46.
  Tokens (17): ['the', 'mal', '##ware', 'communicate', '##s', 'with', 'c2', 'server', 'at', '185', '.', '92', '.', '222', '.', '46', '.']

Sentence: Emotet uses macro-enabled documents for initial access.
  Tokens (12): ['em', '##ote', '##t', 'uses', 'macro', '-', 'enabled', 'documents', 'for', 'initial', 'access', '.']

Sentence: Hash: d41d8cd98f00b204e9800998ecf8427e was observed in the attack.
  Tokens (33): ['hash', ':', 'd', '##41', '##d', '##8', '##cd', '##9', '##8', '##f', '##00', '##b', '##20', '##4', '##e', '##9', '##80', '##0', '##9', '##9', '##8', '##ec', '##f', '##8', '##42', '##7', '##e', 'was', 'observed', 'in', 'the', 'attack', '.']

Sentence: The threat actor exploited CVE-2021-44228 (Log4Shell).
  Tokens (19): ['the', 'threat', 'actor', 'exp

## Summary

- BERT reads **tokens**, not words. A single CTI term may become many tokens.
- `[CLS]` (position 0) represents the whole sentence → used for **classification**.
- Each middle token has its own output vector → used for **NER**.
- `attention_mask` distinguishes real tokens from padding.
- CTI-specific text (hashes, IPs, CVEs, malware names) fragments heavily with general BERT — this motivates domain-adapted models.

## Next notebook

**`02_bert_outputs_cls_vs_tokens.ipynb`** — we'll actually run a BERT model (not just the tokenizer) and inspect its output shape. You'll see concretely why `[CLS]` is used for classification and per-token vectors are used for NER.